In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

### Reading the Table

In [0]:
# Read Silver sales and Gold dims
df_sales = spark.table("dev_project.silver.crm_sales_details")

df_dim_customers = spark.table("dev_project.gold.dim_customers") \
    .select("customer_surrogate_key", "customer_id")

# From dim_products, get one row per prd_key_short
# Prefer is_current = true, fall back to latest version
window_best = Window.partitionBy("prd_key_short") \
                    .orderBy(
                        F.col("is_current").desc(),      # current first
                        F.col("start_date").desc()        # then latest version
                    )

df_dim_products = spark.table("dev_project.gold.dim_products") \
    .withColumn("_rank", F.row_number().over(window_best)) \
    .filter(F.col("_rank") == 1) \
    .select("product_surrogate_key", "prd_key_short") \
    .drop("_rank")

# Join to dims to get surrogate keys
df_fact_sales = df_sales \
    .join(df_dim_products,
          df_sales["product_key"] == df_dim_products["prd_key_short"],
          how="left") \
    .join(df_dim_customers,
          on="customer_id",
          how="left") \
    .select(
        "order_number",
        "product_surrogate_key",
        "customer_surrogate_key",
        "order_date",
        "ship_date",
        "due_date",
        "sales_amount",
        "quantity",
        "unit_price"
    )

### Validation


In [0]:
print("\n" + "="*50)
print(" FACT_SALES VALIDATION")
print("="*50)
print(f"Total records:            {df_fact_sales.count():,}")
print(f"Null product surr. keys:  {df_fact_sales.filter(F.col('product_surrogate_key').isNull()).count()}")
print(f"Null customer surr. keys: {df_fact_sales.filter(F.col('customer_surrogate_key').isNull()).count()}")
print(f"Null order numbers:       {df_fact_sales.filter(F.col('order_number').isNull()).count()}")
print(f"Null sales amounts:       {df_fact_sales.filter(F.col('sales_amount').isNull()).count()}")
print(f"Total sales amount:       {df_fact_sales.agg(F.sum('sales_amount')).collect()[0][0]:,}")
print("="*50)

### Writting in to the gold layer

In [0]:
df_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dev_project.gold.fact_sales")

print("fact_sales Gold write complete.")